<a href="https://colab.research.google.com/github/walk0504/DeepLearningIndustryAIChallenge/blob/main/Amazon_Review_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q datasets transformers  accelerate scikit-learn

In [2]:

from datasets import load_dataset

# Load the English Amazon customer reviews
dataset = load_dataset(
    "goosmanlei/amazon_reviews_multi",
    "en"
)

# Check the dataset sizes
print(dataset)

# Display an example review
print("\nExample review:")
print(dataset["train"][0]["review_body"])

print("\nStar rating:")
print(dataset["train"][0]["stars"])


README.md:   0%|          | 0.00/2.62k [00:00<?, ?B/s]

en/train.jsonl.gz: reconstructing file:   0%|          |  0.00B / 21.5MB            

en/train.jsonl.gz: downloading bytes:           |  0.00B            

en/validation.jsonl.gz: reconstructing file:   0%|          |  0.00B /  542kB            

en/validation.jsonl.gz: downloading bytes:           |  0.00B            

en/test.jsonl.gz: reconstructing file:   0%|          |  0.00B /  536kB            

en/test.jsonl.gz: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/200000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 5000
    })
})

Example review:
Arrived broken. Manufacturer defect. Two of the legs of the base were not completely formed, so there was no way to insert the casters. I unpackaged the entire chair and hardware before noticing this. So, I'll spend twice the amount of time boxing up the whole useless thing and send it back with a 1-star review of part of a chair I never got to sit in. I will go so far as to include a picture of what t

In [3]:

import pandas as pd

# Count how many reviews have each star rating
for split in ["train", "validation", "test"]:
    counts = pd.Series(dataset[split]["stars"]).value_counts().sort_index()

    print(f"\n{split.upper()} STAR RATINGS:")
    print(counts)



TRAIN STAR RATINGS:
1    40000
2    40000
3    40000
4    40000
5    40000
Name: count, dtype: int64

VALIDATION STAR RATINGS:
1    1000
2    1000
3    1000
4    1000
5    1000
Name: count, dtype: int64

TEST STAR RATINGS:
1    1000
2    1000
3    1000
4    1000
5    1000
Name: count, dtype: int64


In [4]:

import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


GPU available: True
GPU name: Tesla T4


In [5]:

from datasets import DatasetDict

# Turn star ratings into positive and negative labels
def prepare_review(example):
    example["label"] = 0 if example["stars"] <= 2 else 1
    return example

# Remove neutral (3-star) reviews
binary_dataset = DatasetDict()

for split in ["train", "validation", "test"]:
    reviews = dataset[split].filter(
        lambda example: example["stars"] != 3
    )
    binary_dataset[split] = reviews.map(prepare_review)

# Check the results
for split in binary_dataset:
    reviews = binary_dataset[split]
    negative = reviews["label"].count(0)
    positive = reviews["label"].count(1)

    print(f"{split}: {len(reviews)} reviews")
    print(f"  Negative: {negative}")
    print(f"  Positive: {positive}")


Filter:   0%|          | 0/200000 [00:00<?, ? examples/s]

Map:   0%|          | 0/160000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

train: 160000 reviews
  Negative: 80000
  Positive: 80000
validation: 4000 reviews
  Negative: 2000
  Positive: 2000
test: 4000 reviews
  Negative: 2000
  Positive: 2000


In [6]:

# Check for duplicate review text within and across splits
review_sets = {}

for split in ["train", "validation", "test"]:
    texts = [text.strip().lower() for text in binary_dataset[split]["review_body"]]
    unique_texts = set(texts)
    review_sets[split] = unique_texts

    print(f"{split}: {len(texts) - len(unique_texts)} duplicate review texts")

print("\nDuplicates across splits:")
print("Train / Validation:", len(review_sets["train"] & review_sets["validation"]))
print("Train / Test:", len(review_sets["train"] & review_sets["test"]))
print("Validation / Test:", len(review_sets["validation"] & review_sets["test"]))


train: 563 duplicate review texts
validation: 1 duplicate review texts
test: 0 duplicate review texts

Duplicates across splits:
Train / Validation: 19
Train / Test: 18
Validation / Test: 1


In [7]:

from datasets import DatasetDict

# Create a consistent version of each review for checking duplicates
def add_clean_text(example):
    example["clean_text"] = " ".join(
        example["review_body"].lower().split()
    )
    return example

cleaned = DatasetDict()

for split in ["train", "validation", "test"]:
    cleaned[split] = binary_dataset[split].map(add_clean_text)

# Work from test backward to prevent overlap
seen = set()

for split in ["test", "validation", "train"]:
    original_count = len(cleaned[split])

    # Remove repeated review text and overlaps with protected splits
    keep_indices = []

    for i, text in enumerate(cleaned[split]["clean_text"]):
        if text not in seen:
            keep_indices.append(i)
            seen.add(text)

    cleaned[split] = cleaned[split].select(keep_indices)

    print(
        f"{split}: {original_count} -> {len(cleaned[split])} reviews"
    )

# Check that no review text appears across splits
assert not (
    set(cleaned["train"]["clean_text"])
    & set(cleaned["validation"]["clean_text"])
)

assert not (
    set(cleaned["train"]["clean_text"])
    & set(cleaned["test"]["clean_text"])
)

assert not (
    set(cleaned["validation"]["clean_text"])
    & set(cleaned["test"]["clean_text"])
)

print("\nDuplicate check passed!")


Map:   0%|          | 0/160000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

test: 4000 -> 4000 reviews
validation: 4000 -> 3998 reviews
train: 160000 -> 159401 reviews

Duplicate check passed!


In [8]:

# Check class balance after removing duplicates

for split in ["train", "validation", "test"]:
    labels = pd.Series(cleaned[split]["label"])

    print(f"\n{split.upper()}")
    print("Negative reviews:", (labels == 0).sum())
    print("Positive reviews:", (labels == 1).sum())
    print("Total reviews:", len(labels))



TRAIN
Negative reviews: 79759
Positive reviews: 79642
Total reviews: 159401

VALIDATION
Negative reviews: 1999
Positive reviews: 1999
Total reviews: 3998

TEST
Negative reviews: 2000
Positive reviews: 2000
Total reviews: 4000


In [9]:

from datasets import DatasetDict, concatenate_datasets

# Select an equal number of positive and negative reviews
def balanced_sample(data, total, seed=42):
    negatives = data.filter(lambda x: x["label"] == 0)
    positives = data.filter(lambda x: x["label"] == 1)

    negatives = negatives.shuffle(seed=seed).select(range(total // 2))
    positives = positives.shuffle(seed=seed).select(range(total // 2))

    return concatenate_datasets(
        [negatives, positives]
    ).shuffle(seed=seed)

# Temporary dataset for measuring training speed
speed_test_data = DatasetDict({
    "train": balanced_sample(cleaned["train"], 1000),
    "validation": balanced_sample(cleaned["validation"], 200)
})

print("Speed test training reviews:", len(speed_test_data["train"]))
print("Speed test validation reviews:", len(speed_test_data["validation"]))


Filter:   0%|          | 0/159401 [00:00<?, ? examples/s]

Filter:   0%|          | 0/159401 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3998 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3998 [00:00<?, ? examples/s]

Speed test training reviews: 1000
Speed test validation reviews: 200


In [10]:

from transformers import AutoTokenizer

# Choose our pre-trained model
model_name = "distilbert-base-uncased"

# Load its tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Convert review text into numbers the model understands
def tokenize_reviews(batch):
    return tokenizer(
        batch["review_body"],
        truncation=True,
        max_length=128
    )

# Prepare our temporary speed-test dataset
tokenized_speed_data = speed_test_data.map(
    tokenize_reviews,
    batched=True
)

print("Tokenization complete!")
print("Training reviews:", len(tokenized_speed_data["train"]))
print("Validation reviews:", len(tokenized_speed_data["validation"]))


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenization complete!
Training reviews: 1000
Validation reviews: 200


In [11]:

import time
import torch
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# Load pre-trained DistilBERT for positive/negative reviews
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Freeze the original DistilBERT layers
# Only train the classification layers
for param in model.distilbert.parameters():
    param.requires_grad = False

# Automatically pad reviews to the right length
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Training settings for our speed test
training_args = TrainingArguments(
    output_dir="./speed_test_results",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    report_to="none",
    seed=42
)

# Set up the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_speed_data["train"],
    eval_dataset=tokenized_speed_data["validation"],
    data_collator=data_collator,
    processing_class=tokenizer
)

# Measure how long training takes
start_time = time.time()

trainer.train()

elapsed = time.time() - start_time

print(f"\nTraining time: {elapsed:.1f} seconds")
print(f"Training time: {elapsed / 60:.2f} minutes")
print("GPU:", torch.cuda.get_device_name(0))


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.592020,0.588657



Training time: 4.9 seconds
Training time: 0.08 minutes
GPU: Tesla T4


In [12]:

# Create our proposed dataset for the official experiments

experiment_data = DatasetDict({
    "train": balanced_sample(cleaned["train"], 16000),
    "validation": balanced_sample(cleaned["validation"], 2000),
    "test": cleaned["test"]
})

# Display dataset sizes
for split in experiment_data:
    print(f"{split}: {len(experiment_data[split])} reviews")

print("\nExperiment dataset created!")


train: 16000 reviews
validation: 2000 reviews
test: 4000 reviews

Experiment dataset created!


In [13]:

# Convert our experiment reviews into tokens
tokenized_experiment_data = experiment_data.map(
    tokenize_reviews,
    batched=True
)

# Check that everything worked
for split in tokenized_experiment_data:
    print(f"{split}: {len(tokenized_experiment_data[split])} reviews")

print("\nTokenization complete!")


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

train: 16000 reviews
validation: 2000 reviews
test: 4000 reviews

Tokenization complete!
